# 4.2. Metrics: Similarity (Word Overlap, Cosine Similarity, BLEU)

In [1]:
import pandas as pd

In [2]:
# Reading the data from the file
df = pd.read_csv("../data/raw/paranmt_for_detox_500k.tsv", sep="\t", index_col=0)
df.head()

,reference,translation,similarity,length_diff,ref_tox,trn_tox
0,"If Alkar is flooding her with psychic waste, t...","if Alkar floods her with her mental waste, it ...",0.785171,0.010309,0.014195,0.981983
1,Now you're getting nasty.,you're becoming disgusting.,0.749687,0.071429,0.065473,0.999039
2,"Well, we could spare your life, for one.","well, we can spare your life.",0.919051,0.268293,0.213313,0.985068
3,"Ah! Monkey, you've got to snap out of it.","monkey, you have to wake up.",0.664333,0.309524,0.053362,0.994215
4,I've got orders to put her down.,I have orders to kill her.,0.726639,0.181818,0.009402,0.999348


## 1. Word Overlap

In [3]:
import string


def get_wo_score(ref: str, hyp: str) -> float:
    """
    Returns the Word Overlap score.
    """
    # Remove all punctuation and replace with whitespace
    ref = ref.translate(
        str.maketrans(string.punctuation, " " * len(string.punctuation))
    )
    hypothesis = hyp.translate(
        str.maketrans(string.punctuation, " " * len(string.punctuation))
    )

    # Calculate the number of words in the reference and hypothesis
    ref_words = set(ref.lower().split())
    hyp_words = set(hyp.lower().split())

    inter_len = len(ref_words.intersection(hyp_words))
    union_len = len(ref_words.union(hyp_words))

    # Avoid division by zero
    if union_len == 0:
        return 0

    return inter_len / union_len

## 2. Cosine Similarity

In [4]:
import spacy

nlp = spacy.load("en_core_web_lg")

def get_cosine_score(ref: str, hyp: str) -> float:
    """
    Returns the Cosine Similarity score.
    """
    ref_doc = nlp(ref)
    hyp_doc = nlp(hyp)

    return ref_doc.similarity(hyp_doc)

## 3. BLEU

In [5]:
import nltk

def get_bleu_score(ref: str, hyp: str) -> float:
    """
    Returns the BLEU score.
    """
    return nltk.translate.bleu_score.sentence_bleu([ref], hyp)

In [6]:
# Compare some random sentences from the dataset
for i in range(10):
    # Get a random index
    index = df.sample().index[0]

    # Get the reference and hypothesis sentences
    reference = df.loc[index, "reference"]
    hypothesis = df.loc[index, "translation"]

    # Calculate scores
    wo_score = get_wo_score(reference, hypothesis)
    cs_score = get_cosine_score(reference, hypothesis)
    bleu_score = get_bleu_score(reference, hypothesis)

    # Print the results
    print("Reference:   ", reference)
    print("Hypothesis:  ", hypothesis)
    print("WO score:    ", wo_score)
    print("Cosine score:", cs_score)
    print("BLEU score:  ", bleu_score)
    print()

Reference:    In a cafe, I called Admiral Darlan a jackass.
Hypothesis:   I told the coffee shop that Admiral Darlan was a lousy defector.
WO score:     0.25
Cosine score: 0.7779576667347232
BLEU score:   0.3172337874290939

Reference:    These guys down on Wall Street,
Hypothesis:   these junkies on Wall Street,
WO score:     0.375
Cosine score: 0.9380057789135487
BLEU score:   0.635361252026423

Reference:    Do you even remember his name? - Shut up.
Hypothesis:   do you even remember his name?
WO score:     0.5555555555555556
Cosine score: 0.8338460144832754
BLEU score:   0.668685203401056

Reference:    "And last--if it helps any, just think of me as a very crazy fellow who went berserk one summer day and never was right again.
Hypothesis:   "and one more thing - if it's any good, I'm like a foolish man who got a frenzy of fury on a summer day, and he never recovered.
WO score:     0.21428571428571427
Cosine score: 0.9474940681057611
BLEU score:   0.4155504154842616

Reference:    